In [46]:
import os
import librosa
import numpy as np
from sklearn.model_selection import train_test_split
import soundfile as sf

In [47]:
# Diretórios principais
CHORUS_DIR = '../../data/Chorus/Chorus'
HALL_REVERB_DIR = '../../data/Hall-Reverb/Hall-Reverb'
OUTPUT_DIR = '../../data/output'

print(os.path.abspath(CHORUS_DIR))
print(os.path.abspath(HALL_REVERB_DIR))
print(os.path.abspath(OUTPUT_DIR))

# Parâmetros do Mel-Spectrograma
SAMPLE_RATE = 22050
N_MELS = 128
N_FFT = 2048
HOP_LENGTH = 512

c:\utfpr\TCC\tone-capture\data\Chorus\Chorus
c:\utfpr\TCC\tone-capture\data\Hall-Reverb\Hall-Reverb
c:\utfpr\TCC\tone-capture\data\output


In [48]:
def process_audio_file(file_path):
    """Carrega um arquivo de áudio e retorna o mel-espectrograma."""
    y, sr = librosa.load(file_path, sr=SAMPLE_RATE)
    mel_spec = librosa.feature.melspectrogram(y=y, sr=sr, n_fft=N_FFT, hop_length=HOP_LENGTH, n_mels=N_MELS)
    mel_spec_db = librosa.power_to_db(mel_spec, ref=np.max)
    return mel_spec_db

In [49]:
def pair_files(chorus_dir, hall_reverb_dir):
    """Emparelha arquivos de Chorus e Hall-Reverb com base nos nomes."""
    paired_files = []
    for root, _, files in os.walk(hall_reverb_dir):
        for file in files:
            if file.endswith('.wav'):
                hall_path = os.path.join(root, file)
                chorus_path = hall_path.replace(hall_reverb_dir, chorus_dir)
                if os.path.exists(chorus_path):
                    paired_files.append((hall_path, chorus_path))
    return paired_files

In [50]:
def process_and_save_pairs(paired_files, output_dir):
    """Processa os pares de arquivos e salva os espectrogramas e nomes dos arquivos."""
    input_specs = []
    target_specs = []
    input_file_names = []
    target_file_names = []
    
    for hall_path, chorus_path in paired_files:
        # Processar arquivos
        input_spec = process_audio_file(hall_path)
        target_spec = process_audio_file(chorus_path)
        
        input_specs.append(input_spec)
        target_specs.append(target_spec)
        
        # Salvar nomes dos arquivos para mapeamento
        input_file_names.append(hall_path)
        target_file_names.append(chorus_path)
    
    # Criar o diretório de saída se não existir
    os.makedirs(output_dir, exist_ok=True)
    
    # Salvar espectrogramas e nomes dos arquivos
    np.save(os.path.join(output_dir, 'input_specs.npy'), np.array(input_specs))
    np.save(os.path.join(output_dir, 'target_specs.npy'), np.array(target_specs))
    np.save(os.path.join(output_dir, 'input_file_names.npy'), np.array(input_file_names))
    np.save(os.path.join(output_dir, 'target_file_names.npy'), np.array(target_file_names))


In [51]:
# Emparelhamento e processamento
paired_files = pair_files(CHORUS_DIR, HALL_REVERB_DIR)
process_and_save_pairs(paired_files, OUTPUT_DIR)

In [52]:
def split_data(input_specs, target_specs, input_file_names, target_file_names, test_size=0.2):
    """Divide os dados em treino, validação e teste mantendo o mapeamento de nomes."""
    input_train, input_test, target_train, target_test, input_names_train, input_names_test, target_names_train, target_names_test = train_test_split(
        input_specs, target_specs, input_file_names, target_file_names, test_size=test_size, random_state=42)
    input_val, input_test, target_val, target_test, input_names_val, input_names_test, target_names_val, target_names_test = train_test_split(
        input_test, target_test, input_names_test, target_names_test, test_size=0.5, random_state=42)
    return (input_train, input_val, input_test, 
            target_train, target_val, target_test, 
            input_names_train, input_names_val, input_names_test, 
            target_names_train, target_names_val, target_names_test)

# Carregar espectrogramas e nomes dos arquivos
input_specs = np.load(os.path.join(OUTPUT_DIR, 'input_specs.npy'))
target_specs = np.load(os.path.join(OUTPUT_DIR, 'target_specs.npy'))
input_file_names = np.load(os.path.join(OUTPUT_DIR, 'input_file_names.npy'))
target_file_names = np.load(os.path.join(OUTPUT_DIR, 'target_file_names.npy'))

# Dividir os dados
(input_train, input_val, input_test, 
 target_train, target_val, target_test, 
 input_names_train, input_names_val, input_names_test, 
 target_names_train, target_names_val, target_names_test) = split_data(
    input_specs, target_specs, input_file_names, target_file_names)

# Salvar conjuntos divididos
np.save(os.path.join(OUTPUT_DIR, 'train_input.npy'), input_train)
np.save(os.path.join(OUTPUT_DIR, 'val_input.npy'), input_val)
np.save(os.path.join(OUTPUT_DIR, 'test_input.npy'), input_test)
np.save(os.path.join(OUTPUT_DIR, 'train_target.npy'), target_train)
np.save(os.path.join(OUTPUT_DIR, 'val_target.npy'), target_val)
np.save(os.path.join(OUTPUT_DIR, 'test_target.npy'), target_test)

# Salvar os nomes dos arquivos correspondentes
np.save(os.path.join(OUTPUT_DIR, 'train_input_names.npy'), input_names_train)
np.save(os.path.join(OUTPUT_DIR, 'val_input_names.npy'), input_names_val)
np.save(os.path.join(OUTPUT_DIR, 'test_input_names.npy'), input_names_test)
np.save(os.path.join(OUTPUT_DIR, 'train_target_names.npy'), target_names_train)
np.save(os.path.join(OUTPUT_DIR, 'val_target_names.npy'), target_names_val)
np.save(os.path.join(OUTPUT_DIR, 'test_target_names.npy'), target_names_test)


In [56]:
def reconstruct_audio_from_spec(mel_spec, output_wav_path):
    """Reconstrói o áudio a partir de um mel-espectrograma."""
    mel_spec = librosa.db_to_power(mel_spec)
    audio = librosa.feature.inverse.mel_to_audio(mel_spec, sr=SAMPLE_RATE, n_fft=N_FFT, hop_length=HOP_LENGTH)
    sf.write(output_wav_path, audio, SAMPLE_RATE)

# Reconstrução de áudio com mapeamento
test_specs = np.load(os.path.join(OUTPUT_DIR, 'test_input.npy'))
test_file_names = np.load(os.path.join(OUTPUT_DIR, 'test_input_names.npy'))
RECONSTRUCT_DIR = '../../data/reconstruct'
os.makedirs(RECONSTRUCT_DIR, exist_ok=True)

for i, (spec, original_file) in enumerate(zip(test_specs, test_file_names)):
    # Verificar se o arquivo é de Chorus ou Hall-Reverb
    if CHORUS_DIR in original_file:
        category = "Chorus"
    elif HALL_REVERB_DIR in original_file:
        category = "Hall-Reverb"
    else:
        category = "Unknown"  # Apenas para garantir contra arquivos inesperados
    
    # Obter o nome do arquivo e a subpasta de origem
    relative_path = os.path.relpath(original_file, HALL_REVERB_DIR if category == "Hall-Reverb" else CHORUS_DIR)
    folder_name = os.path.dirname(relative_path).replace(os.sep, '_')  # Nome da subpasta (substituir separador por "_")
    base_name = os.path.splitext(os.path.basename(original_file))[0]  # Nome base do arquivo
    
    # Criar o nome de saída com categoria, pasta e arquivo
    output_wav_path = os.path.join(RECONSTRUCT_DIR, f'{category}_{folder_name}_{base_name}_reconstructed.wav')
    
    # Reconstruir o áudio
    reconstruct_audio_from_spec(spec, output_wav_path)
